# Wilcoxon signed-rank: fusion vs image-only (one-sided)

Tests **test macro-F1** and **test accuracy** on 30 matched seeds.

In [1]:
import json
import os
import re
import statistics
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(12):
    if (_p / "experiments").is_dir() and (_p / "data").is_dir():
        PROJECT_ROOT = _p
        break
    _p = _p.parent
else:
    PROJECT_ROOT = Path.cwd().resolve()
os.chdir(PROJECT_ROOT)

from scipy.stats import wilcoxon

FUSION_EXP = "phase3_robustness"
IMAGE_EXP = "imageonly_robustness"
ALPHA = 0.05

METRICS = [
    ("test_macro_f1", "Test Macro F1", False),
    ("test_accuracy", "Test Accuracy", True),
]


def load_test_metric(exp_name: str, json_key: str, as_fraction: bool = False) -> dict[int, float]:
    out: dict[int, float] = {}
    base = PROJECT_ROOT / "experiments" / exp_name / "metrics" / "experiments"
    for path in base.glob("seed_*_results.json"):
        m = re.match(r"seed_(\d+)_results\.json$", path.name)
        if not m:
            continue
        obj = json.loads(path.read_text(encoding="utf-8"))
        v = float(obj["test_metrics"][json_key])
        if as_fraction and v > 1.0:
            v /= 100.0
        out[int(m.group(1))] = v
    return out


for json_key, label, as_fraction in METRICS:
    fusion = load_test_metric(FUSION_EXP, json_key, as_fraction)
    image = load_test_metric(IMAGE_EXP, json_key, as_fraction)
    seeds = sorted(fusion.keys() & image.keys())
    if fusion.keys() != image.keys():
        raise ValueError(f"{label} seed mismatch")

    d = [fusion[s] - image[s] for s in seeds]
    mean_d = statistics.mean(d)
    std_d = statistics.stdev(d) if len(d) > 1 else float("nan")
    cohen_d = mean_d / std_d if std_d else float("nan")
    wins = sum(1 for x in d if x > 0)
    ties = sum(1 for x in d if x == 0)
    losses = sum(1 for x in d if x < 0)

    res = wilcoxon(d, alternative="greater", zero_method="wilcox")

    print("=" * 60)
    print(label)
    print("fusion_mean", statistics.mean(fusion.values()))
    print("image_mean", statistics.mean(image.values()))
    print("n", len(d))
    print("mean_diff", mean_d)
    print("stdev_diff", std_d)
    print("paired_Cohen_d", cohen_d)
    print("wins_ties_losses", wins, ties, losses)
    print("median_diff", statistics.median(d))
    print("statistic", res.statistic)
    print("pvalue", res.pvalue)
    print("reject_H0_fusion_gt_image", res.pvalue < ALPHA)
    print()

Test Macro F1
fusion_mean 0.8310843219571423
image_mean 0.8145000550248405
n 30
mean_diff 0.016584266932301864
stdev_diff 0.010423274665449109
paired_Cohen_d 1.5910802952622085
wins_ties_losses 27 0 3
median_diff 0.017163615028558676
statistic 457.0
pvalue 2.3283064365386963e-08
reject_H0_fusion_gt_image True

Test Accuracy
fusion_mean 0.831183879093199
image_mean 0.8152549094359614
n 30
mean_diff 0.015928969657237675
stdev_diff 0.010019102868218463
paired_Cohen_d 1.589859877351481
wins_ties_losses 27 0 3
median_diff 0.017104928376968698
statistic 458.0
pvalue 1.7695128917694092e-08
reject_H0_fusion_gt_image True

